# Woche 2 – Tag 4: Reproduzierbare Datenpipeline

## 1. Fehleranalyse des Chaos-Notebooks

| Fundstelle | Problem | Geplante Lösung |
|---|---|---|
| In [9] | Export steht am Anfang und überschreibt die Rohdatei | Export ans Ende verschieben; `raw/` und `clean/` trennen |
| In [4] | Preisumwandlung berücksichtigt nicht alle Zahlenformate | Tausendertrennzeichen entfernen und Preise kontrolliert konvertieren |
| In [3] | `drop_duplicates()` entfernt möglicherweise echte Bestellungen | Duplikate anhand geeigneter Schlüssel prüfen |
| In [7] | `Timestamp.today()` erzeugt einen veränderlichen Stichtag | Festen Stichtag verwenden |
| In [5] | Nicht erkannte Datumsformate werden zu `NaT` | Datumsformate prüfen und ungültige Werte dokumentieren |
| In [6] | Unvollständiges Länder-Mapping erzeugt fehlende Werte | Schreibweisen normalisieren und unbekannte Werte prüfen |
| Markdown-Kommentar | Rabatte wurden manuell in Excel verändert | Rabattbereinigung im Python-Skript durchführen |
| In [8] | `Timestamp.now()` verändert bei jedem Lauf die Ausgabe | Veränderlichen Zeitstempel aus den Exportdaten entfernen |
| In [2] | Pandas-Version ist nicht festgelegt | Abhängigkeiten in `requirements.txt` mit festen Versionen dokumentieren |

## 2. Ziel der neuen Pipeline

Die neue Pipeline soll:

1. Rohdaten unverändert lassen.
2. Alle Bereinigungsschritte automatisch ausführen.
3. Einen festen Zeitraum verwenden.
4. Bereinigte Daten sortiert im Ordner `clean/` speichern.
5. Bei identischen Eingabedaten identische Ergebnisse erzeugen.
6. Die Reproduzierbarkeit mit SHA-256 überprüfen.

## 3. Bereitstellung und Verarbeitung der Rohdaten

Die Datei `bestellungen_april_roh.csv` wurde von Laurin nachgereicht.

Die Rohdaten liegen unverändert im Ordner `raw/`.

Die Pipeline bereinigt die Daten automatisch und speichert
71 April-Bestellungen im Ordner `clean/`.

Der Test in einem frischen Verzeichnis war erfolgreich.
Beide Durchläufe erzeugten denselben SHA-256-Hash.


In [ ]:
import pandas as pd
from pathlib import Path

datei = Path.home() / "Desktop" / "Modul 5" / "Aufgaben" / "Woche_2" / "bestellungen_april_roh.csv"

df = pd.read_csv(datei)

print("Zeilen und Spalten:", df.shape)
print("Spalten:", df.columns.tolist())
display(df.head())

In [ ]:
print("=== DATENTYPEN ===")
print(df.dtypes)

print("\n=== FEHLENDE WERTE ===")
print(df.isna().sum())

print("\n=== VOLLSTÄNDIGE DUPLIKATE ===")
print(df.duplicated().sum())

print("\n=== LÄNDERWERTE ===")
print(df["land"].value_counts(dropna=False))

print("\n=== RABATTWERTE ===")
print(df["rabatt_pct"].value_counts(dropna=False))

In [ ]:
print("Gesamtzahl der Zeilen:", len(df))
print("Eindeutige Bestell-IDs:", df["bestell_id"].nunique())
print("Doppelte Bestell-IDs:", df["bestell_id"].duplicated().sum())

print("\n=== BESTELLUNGEN MIT DOPPELTEN IDs ===")
display(
    df[df["bestell_id"].duplicated(keep=False)]
    .sort_values("bestell_id")
    .drop(columns=["kunde_email"])
)

In [ ]:
# Arbeitskopie erstellen – Rohdaten bleiben unverändert
df_clean = df.copy()

# Vollständig identische Zeilen entfernen
df_clean = df_clean.drop_duplicates().copy()

print("Zeilen vorher:", len(df))
print("Zeilen nachher:", len(df_clean))
print("Entfernte Duplikate:", len(df) - len(df_clean))

# Sicherstellen, dass keine vollständigen Duplikate übrig sind
assert df_clean.duplicated().sum() == 0

print("✅ Duplikatprüfung bestanden")

In [ ]:
# Beide Datumsformate getrennt erkennen
datum_iso = pd.to_datetime(
    df_clean["datum"],
    format="%Y-%m-%d",
    errors="coerce"
)

datum_de = pd.to_datetime(
    df_clean["datum"],
    format="%d.%m.%Y",
    errors="coerce"
)

# Ergebnisse zusammenführen
df_clean["datum_parsed"] = datum_iso.fillna(datum_de)

print("Bestellungen insgesamt:", len(df_clean))
print("Erfolgreich erkannte Datumswerte:", df_clean["datum_parsed"].notna().sum())
print("Nicht erkannte Datumswerte:", df_clean["datum_parsed"].isna().sum())

print("\nBeispiele:")
display(df_clean[["datum", "datum_parsed"]].head(10))

In [ ]:
print("=== PREISFORMATE ===")

preise = df_clean["preis_eur"].astype("string")

print("Preise mit Komma:", preise.str.contains(",", regex=False).sum())
print("Preise mit Punkt:", preise.str.contains(".", regex=False).sum())

print("\n=== BEISPIELE ===")
print("Mit Komma:")
print(preise[preise.str.contains(",", regex=False)].unique()[:10])

print("\nMit Punkt:")
print(preise[preise.str.contains(".", regex=False)].unique()[:10])

print("\n=== FEHLENDE PREISE ===")
print(preise.isna().sum())

In [ ]:
# Preise vereinheitlichen
df_clean["preis_eur"] = pd.to_numeric(
    df_clean["preis_eur"]
    .astype("string")
    .str.replace(",", ".", regex=False),
    errors="raise"
)

print("Datentyp:", df_clean["preis_eur"].dtype)
print("Fehlende Preise:", df_clean["preis_eur"].isna().sum())

print("\nBereinigte Preise:")
print(sorted(df_clean["preis_eur"].unique()))

assert df_clean["preis_eur"].notna().all()

print("\n✅ Preisbereinigung erfolgreich!")

In [ ]:
# Rabattwerte vereinheitlichen
df_clean["rabatt_pct"] = pd.to_numeric(
    df_clean["rabatt_pct"]
    .astype("string")
    .str.replace("%", "", regex=False)
    .str.strip(),
    errors="raise"
)

print("Datentyp:", df_clean["rabatt_pct"].dtype)
print("Fehlende Rabatte:", df_clean["rabatt_pct"].isna().sum())

print("\nBereinigte Rabattwerte:")
print(sorted(df_clean["rabatt_pct"].unique()))

# Qualitätskontrolle
assert df_clean["rabatt_pct"].notna().all()
assert df_clean["rabatt_pct"].between(0, 100).all()

print("\n✅ Rabattbereinigung erfolgreich!")

In [ ]:
# Ländernamen vereinheitlichen
laender_mapping = {
    "DE": "DE",
    "de": "DE",
    "Deutschland": "DE",
    "AT": "AT",
    "Österreich": "AT"
}

df_clean["land"] = df_clean["land"].map(laender_mapping)

print("=== BEREINIGTE LÄNDER ===")
print(df_clean["land"].value_counts(dropna=False))

# Qualitätskontrolle
assert df_clean["land"].notna().all()
assert df_clean["land"].isin(["DE", "AT"]).all()

print("\n✅ Länderbereinigung erfolgreich!")

In [ ]:
# Fester Berichtszeitraum: April 2026
startdatum = pd.Timestamp("2026-04-01")
enddatum = pd.Timestamp("2026-05-01")

# Bestellungen innerhalb des Zeitraums auswählen
df_clean = df_clean[
    (df_clean["datum_parsed"] >= startdatum) &
    (df_clean["datum_parsed"] < enddatum)
].copy()

print("Bestellungen im April 2026:", len(df_clean))

print("Frühestes Datum:", df_clean["datum_parsed"].min())
print("Spätestes Datum:", df_clean["datum_parsed"].max())

# Qualitätskontrolle
assert df_clean["datum_parsed"].between(
    startdatum, enddatum, inclusive="left"
).all()

print("\n✅ Fester April-Zeitraum erfolgreich angewendet!")

In [ ]:
# Bestellungen außerhalb des April-Zeitraums ermitteln
ausgeschlossen = df.loc[
    ~df["bestell_id"].isin(df_clean["bestell_id"])
].drop_duplicates().copy()

print("Ausgeschlossene Bestellungen:", len(ausgeschlossen))

# Ohne personenbezogene Daten anzeigen
display(
    ausgeschlossen[
        ["bestell_id", "datum", "produkt", "menge", "preis_eur"]
    ].sort_values("bestell_id")
)

In [ ]:
# Abschließende Qualitätskontrolle

assert len(df_clean) == 71, "Unerwartete Anzahl an Bestellungen"
assert df_clean["bestell_id"].is_unique, "Doppelte Bestell-IDs vorhanden"
assert df_clean["datum_parsed"].notna().all(), "Ungültige Datumswerte"
assert df_clean["preis_eur"].notna().all(), "Fehlende Preise"
assert df_clean["rabatt_pct"].between(0, 100).all(), "Ungültige Rabatte"
assert df_clean["land"].isin(["DE", "AT"]).all(), "Unbekannte Länder"

print("✅ Alle Qualitätsprüfungen bestanden!")
print("Bereinigte April-Bestellungen:", len(df_clean))
print("Fehlende Kunden-E-Mails:", df_clean["kunde_email"].isna().sum())

In [ ]:
from pathlib import Path

# Separater Arbeitsordner für die Pipeline
projekt = Path.home() / "Desktop" / "Modul 5" / "Aufgaben" / "Woche_2" / "Pipeline_Tag4"

# Ordner erstellen
(projekt / "raw").mkdir(parents=True, exist_ok=True)
(projekt / "clean").mkdir(parents=True, exist_ok=True)

print("Projektordner:", projekt)
print("Raw-Ordner:", (projekt / "raw").exists())
print("Clean-Ordner:", (projekt / "clean").exists())

In [ ]:
import shutil

# Originaldatei und Ziel festlegen
quelle = Path.home() / "Desktop" / "Modul 5" / "Aufgaben" / "Woche_2" / "bestellungen_april_roh.csv"
ziel = projekt / "raw" / "bestellungen_april_roh.csv"

# Rohdatei kopieren
shutil.copy2(quelle, ziel)

print("Rohdatei vorhanden:", ziel.exists())
print("Dateiname:", ziel.name)
print("Dateigröße:", ziel.stat().st_size, "Bytes")

In [ ]:
# Grundgerüst unseres Python-Skripts erstellen

skript = projekt / "clean_bestellungen.py"

inhalt = '''from pathlib import Path
import pandas as pd

# Projektverzeichnis automatisch ermitteln
BASE_DIR = Path(__file__).resolve().parent

# Eingabe- und Ausgabedateien
RAW_DATEI = BASE_DIR / "raw" / "bestellungen_april_roh.csv"
CLEAN_DATEI = BASE_DIR / "clean" / "bestellungen_april_clean.csv"

# Fester Berichtszeitraum
STARTDATUM = pd.Timestamp("2026-04-01")
ENDDATUM = pd.Timestamp("2026-05-01")


def main():
    print("Starte Bereinigung der April-Bestellungen ...")

    # Rohdaten einlesen
    df = pd.read_csv(RAW_DATEI)

    print(f"Eingelesene Bestellungen: {len(df)}")


if __name__ == "__main__":
    main()
'''

skript.write_text(inhalt, encoding="utf-8")

print("Skript erstellt:", skript.exists())
print("Dateiname:", skript.name)

In [ ]:
# Bestehendes Skript einlesen
inhalt = skript.read_text(encoding="utf-8")

# Duplikatbereinigung nach dem Einlesen ergänzen
alte_stelle = '    print(f"Eingelesene Bestellungen: {len(df)}")'

neue_stelle = '''    print(f"Eingelesene Bestellungen: {len(df)}")

    # Vollständig identische Bestellungen entfernen
    df = df.drop_duplicates().copy()

    print(f"Bestellungen nach Duplikatbereinigung: {len(df)}")
'''

assert inhalt.count(alte_stelle) == 1, "Einfügestelle nicht eindeutig gefunden"

inhalt = inhalt.replace(alte_stelle, neue_stelle)

# Aktualisiertes Skript speichern
skript.write_text(inhalt, encoding="utf-8")

print("✅ Duplikatbereinigung im Skript ergänzt!")

In [ ]:
import subprocess
import sys

# Python-Skript unabhängig vom Notebook ausführen
ergebnis = subprocess.run(
    [sys.executable, str(skript)],
    capture_output=True,
    text=True,
    cwd=projekt
)

print("=== SKRIPT-AUSGABE ===")
print(ergebnis.stdout)

if ergebnis.stderr:
    print("=== FEHLERMELDUNGEN ===")
    print(ergebnis.stderr)

print("Exit-Code:", ergebnis.returncode)

In [ ]:
# Bestehendes Skript einlesen
inhalt = skript.read_text(encoding="utf-8")

# Einfügestelle festlegen
alte_stelle = '    print(f"Bestellungen nach Duplikatbereinigung: {len(df)}")'

neue_stelle = '''    print(f"Bestellungen nach Duplikatbereinigung: {len(df)}")

    # Unterschiedliche Datumsformate vereinheitlichen
    datum_iso = pd.to_datetime(
        df["datum"], format="%Y-%m-%d", errors="coerce"
    )

    datum_de = pd.to_datetime(
        df["datum"], format="%d.%m.%Y", errors="coerce"
    )

    df["datum_parsed"] = datum_iso.fillna(datum_de)

    # Ungültige Datumswerte erkennen
    assert df["datum_parsed"].notna().all(), "Ungültige Datumswerte!"

    print(f"Erfolgreich erkannte Datumswerte: {df['datum_parsed'].notna().sum()}")
'''

assert inhalt.count(alte_stelle) == 1, "Einfügestelle nicht eindeutig gefunden"

inhalt = inhalt.replace(alte_stelle, neue_stelle)

skript.write_text(inhalt, encoding="utf-8")

print("✅ Datumsbereinigung im Skript ergänzt!")

In [ ]:
# Aktualisiertes Python-Skript testen
ergebnis = subprocess.run(
    [sys.executable, str(skript)],
    capture_output=True,
    text=True,
    cwd=projekt
)

print("=== SKRIPT-AUSGABE ===")
print(ergebnis.stdout)

if ergebnis.stderr:
    print("=== FEHLERMELDUNGEN ===")
    print(ergebnis.stderr)

print("Exit-Code:", ergebnis.returncode)

assert ergebnis.returncode == 0, "Skript wurde mit einem Fehler beendet!"

In [ ]:
# Bestehendes Skript einlesen
inhalt = skript.read_text(encoding="utf-8")

# Einfügestelle festlegen
alte_stelle = '    print(f"Erfolgreich erkannte Datumswerte: {df[\'datum_parsed\'].notna().sum()}")'

neue_stelle = '''    print(f"Erfolgreich erkannte Datumswerte: {df['datum_parsed'].notna().sum()}")

    # Preisformate vereinheitlichen
    df["preis_eur"] = pd.to_numeric(
        df["preis_eur"]
        .astype("string")
        .str.replace(",", ".", regex=False),
        errors="raise"
    )

    # Qualitätskontrolle
    assert df["preis_eur"].notna().all(), "Fehlende Preise!"

    print(f"Erfolgreich bereinigte Preise: {df['preis_eur'].notna().sum()}")
'''

assert inhalt.count(alte_stelle) == 1, "Einfügestelle nicht eindeutig gefunden"

inhalt = inhalt.replace(alte_stelle, neue_stelle)

# Skript aktualisieren
skript.write_text(inhalt, encoding="utf-8")

print("✅ Preisbereinigung im Skript ergänzt!")

In [ ]:
ergebnis = subprocess.run(
    [sys.executable, str(skript)],
    capture_output=True,
    text=True,
    cwd=projekt
)

print("=== SKRIPT-AUSGABE ===")
print(ergebnis.stdout)

if ergebnis.stderr:
    print("=== FEHLERMELDUNGEN ===")
    print(ergebnis.stderr)

print("Exit-Code:", ergebnis.returncode)

assert ergebnis.returncode == 0, "Skript wurde mit einem Fehler beendet!"

In [ ]:
# Bestehendes Skript einlesen
inhalt = skript.read_text(encoding="utf-8")

# Einfügestelle festlegen
alte_stelle = '    print(f"Erfolgreich bereinigte Preise: {df[\'preis_eur\'].notna().sum()}")'

neue_stelle = '''    print(f"Erfolgreich bereinigte Preise: {df['preis_eur'].notna().sum()}")

    # Rabattwerte vereinheitlichen
    df["rabatt_pct"] = pd.to_numeric(
        df["rabatt_pct"]
        .astype("string")
        .str.replace("%", "", regex=False)
        .str.strip(),
        errors="raise"
    )

    # Qualitätskontrolle
    assert df["rabatt_pct"].notna().all(), "Fehlende Rabatte!"
    assert df["rabatt_pct"].between(0, 100).all(), "Ungültige Rabatte!"

    print(f"Erfolgreich bereinigte Rabatte: {df['rabatt_pct'].notna().sum()}")
'''

assert inhalt.count(alte_stelle) == 1, "Einfügestelle nicht eindeutig gefunden"

# Skript aktualisieren
inhalt = inhalt.replace(alte_stelle, neue_stelle)
skript.write_text(inhalt, encoding="utf-8")

print("✅ Rabattbereinigung im Skript ergänzt!")

In [ ]:
# Bestehendes Skript einlesen
inhalt = skript.read_text(encoding="utf-8")

# Einfügestelle festlegen
alte_stelle = '    print(f"Erfolgreich bereinigte Rabatte: {df[\'rabatt_pct\'].notna().sum()}")'

neue_stelle = '''    print(f"Erfolgreich bereinigte Rabatte: {df['rabatt_pct'].notna().sum()}")

    # Ländernamen vereinheitlichen
    laender_mapping = {
        "DE": "DE",
        "de": "DE",
        "Deutschland": "DE",
        "AT": "AT",
        "Österreich": "AT"
    }

    df["land"] = df["land"].map(laender_mapping)

    # Qualitätskontrolle
    assert df["land"].notna().all(), "Unbekannte Länderwerte!"
    assert df["land"].isin(["DE", "AT"]).all(), "Ungültige Ländercodes!"

    print(f"Erfolgreich bereinigte Länder: {df['land'].notna().sum()}")
'''

assert inhalt.count(alte_stelle) == 1, "Einfügestelle nicht eindeutig gefunden"

# Skript aktualisieren
inhalt = inhalt.replace(alte_stelle, neue_stelle)
skript.write_text(inhalt, encoding="utf-8")

print("✅ Länderbereinigung im Skript ergänzt!")

In [ ]:
# Aktuelles Python-Skript testen
ergebnis = subprocess.run(
    [sys.executable, str(skript)],
    capture_output=True,
    text=True,
    cwd=projekt
)

print("=== SKRIPT-AUSGABE ===")
print(ergebnis.stdout)

if ergebnis.stderr:
    print("=== FEHLERMELDUNGEN ===")
    print(ergebnis.stderr)

print("Exit-Code:", ergebnis.returncode)

assert ergebnis.returncode == 0, "Skript wurde mit einem Fehler beendet!"

In [ ]:
# Bestehendes Skript einlesen
inhalt = skript.read_text(encoding="utf-8")

# Einfügestelle festlegen
alte_stelle = '    print(f"Erfolgreich bereinigte Länder: {df[\'land\'].notna().sum()}")'

neue_stelle = '''    print(f"Erfolgreich bereinigte Länder: {df['land'].notna().sum()}")

    # Festen April-Zeitraum anwenden
    vorher = len(df)

    df = df[
        (df["datum_parsed"] >= STARTDATUM) &
        (df["datum_parsed"] < ENDDATUM)
    ].copy()

    print(f"Bestellungen außerhalb April: {vorher - len(df)}")
    print(f"Bereinigte April-Bestellungen: {len(df)}")

    # Qualitätskontrolle
    assert len(df) == 71, "Unerwartete Anzahl an April-Bestellungen!"
    assert df["datum_parsed"].between(
        STARTDATUM, ENDDATUM, inclusive="left"
    ).all(), "Bestellungen außerhalb des Berichtszeitraums!"
'''

assert inhalt.count(alte_stelle) == 1, "Einfügestelle nicht eindeutig gefunden"

# Skript aktualisieren
inhalt = inhalt.replace(alte_stelle, neue_stelle)
skript.write_text(inhalt, encoding="utf-8")

print("✅ April-Zeitraum im Skript ergänzt!")

In [ ]:
# Bestehendes Skript einlesen
inhalt = skript.read_text(encoding="utf-8")

# Einfügestelle festlegen
alte_stelle = '''    ).all(), "Bestellungen außerhalb des Berichtszeitraums!"'''

neue_stelle = '''    ).all(), "Bestellungen außerhalb des Berichtszeitraums!"

    # Feste Sortierung für reproduzierbaren Export
    df = df.sort_values("bestell_id").reset_index(drop=True)

    # Datum einheitlich als YYYY-MM-DD speichern
    df["datum"] = df["datum_parsed"].dt.strftime("%Y-%m-%d")
    df = df.drop(columns=["datum_parsed"])

    # Bereinigte CSV speichern
    CLEAN_DATEI.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(
        CLEAN_DATEI,
        index=False,
        encoding="utf-8",
        float_format="%.2f",
        lineterminator="\\n"
    )

    print(f"CSV erfolgreich gespeichert: {CLEAN_DATEI}")
    print(f"Exportierte Bestellungen: {len(df)}")
'''

assert inhalt.count(alte_stelle) == 1, "Einfügestelle nicht eindeutig gefunden"

inhalt = inhalt.replace(alte_stelle, neue_stelle)

skript.write_text(inhalt, encoding="utf-8")

print("✅ CSV-Export im Skript ergänzt!")

In [ ]:
# Vollständige Pipeline ausführen
ergebnis = subprocess.run(
    [sys.executable, str(skript)],
    capture_output=True,
    text=True,
    cwd=projekt
)

print("=== PIPELINE-AUSGABE ===")
print(ergebnis.stdout)

if ergebnis.stderr:
    print("=== FEHLERMELDUNGEN ===")
    print(ergebnis.stderr)

print("Exit-Code:", ergebnis.returncode)

# Qualitätskontrolle
assert ergebnis.returncode == 0, "Pipeline fehlgeschlagen!"

ausgabe = projekt / "clean" / "bestellungen_april_clean.csv"

assert ausgabe.exists(), "Bereinigte CSV wurde nicht erstellt!"

print("\n✅ Pipeline erfolgreich ausgeführt!")
print("✅ Bereinigte CSV vorhanden!")

In [ ]:
import hashlib
import subprocess
import sys

def berechne_sha256(dateipfad):
    return hashlib.sha256(dateipfad.read_bytes()).hexdigest()

hashes = []

for durchlauf in range(1, 3):
    ergebnis = subprocess.run(
        [sys.executable, str(skript)],
        capture_output=True,
        text=True,
        cwd=projekt
    )

    assert ergebnis.returncode == 0, ergebnis.stderr

    datei = projekt / "clean" / "bestellungen_april_clean.csv"
    datei_hash = berechne_sha256(datei)

    hashes.append(datei_hash)

    print(f"Durchlauf {durchlauf}:")
    print(f"SHA-256: {datei_hash}\n")

assert hashes[0] == hashes[1], "Die Ergebnisse unterscheiden sich!"

print("✅ Beide SHA-256-Hashes sind identisch!")
print("✅ Reproduzierbarkeit erfolgreich nachgewiesen!")

In [ ]:
import pandas as pd

# Installierte Pandas-Version ermitteln
pandas_version = pd.__version__

# requirements.txt erstellen
requirements = projekt / "requirements.txt"

requirements.write_text(
    f"pandas=={pandas_version}\n",
    encoding="utf-8"
)

print("Datei erstellt:", requirements.exists())
print("Inhalt:")
print(requirements.read_text(encoding="utf-8"))

In [ ]:
readme = projekt / "README.md"

readme_text = """# Woche 2 – Tag 4: Reproduzierbare Datenpipeline

## Ziel
Automatische Bereinigung der April-Bestellungen.

## Bereinigungsschritte
1. Rohdaten einlesen: 80 Zeilen
2. Vollständige Duplikate entfernen: 6 Zeilen
3. Datumsformate vereinheitlichen
4. Preise bereinigen
5. Rabattwerte bereinigen
6. Ländernamen vereinheitlichen
7. März-Bestellungen ausschließen: 3 Zeilen
8. Daten sortieren und exportieren

## Ergebnis
71 bereinigte April-Bestellungen.

## Projektdateien
- clean_bestellungen.py: Python-Skript
- requirements.txt: Pandas-Version
- raw/: Unveränderte Eingangsdaten
- clean/: Bereinigte Ergebnisse

## Ausführung
1. Abhängigkeiten installieren: python -m pip install -r requirements.txt
2. Pipeline starten: python clean_bestellungen.py

## Reproduzierbarkeit
Zwei Durchläufe erzeugten denselben SHA-256-Hash:

44b8afcc667c74a6da5b765b7732ae34d1c518982b5bd8c0a11f5b1d705862f9

## Datenschutz
Die CSV-Dateien enthalten personenbezogene Daten.
Sie dürfen nicht in ein öffentliches GitHub-Repository hochgeladen werden.
"""

readme.write_text(readme_text, encoding="utf-8")

print("README erstellt:", readme.exists())
print("Dateiname:", readme.name)

In [ ]:
print("=== PROJEKTSTRUKTUR ===")

for datei in sorted(projekt.rglob("*")):
    relativ = datei.relative_to(projekt)

    if datei.is_dir():
        print(f"📁 {relativ}/")
    else:
        print(f"📄 {relativ}")

print("\n=== PFLICHTDATEIEN ===")

pflichtdateien = [
    projekt / "clean_bestellungen.py",
    projekt / "requirements.txt",
    projekt / "README.md",
    projekt / "raw" / "bestellungen_april_roh.csv",
    projekt / "clean" / "bestellungen_april_clean.csv"
]

for datei in pflichtdateien:
    assert datei.exists(), f"Datei fehlt: {datei.name}"

print("✅ Alle benötigten Dateien vorhanden!")

In [ ]:
import shutil

# Abgabeordner vorbereiten
abgabe = projekt.parent / "tag-09"
abgabe.mkdir(parents=True, exist_ok=True)

# Nur die drei vorgesehenen Dateien kopieren
dateien = [
    "clean_bestellungen.py",
    "requirements.txt",
    "README.md"
]

for name in dateien:
    shutil.copy2(projekt / name, abgabe / name)

print("=== ABGABEORDNER ===")

for datei in sorted(abgabe.iterdir()):
    print(datei.name)

print("\n✅ Drei Projektdateien für die Abgabe vorbereitet!")

In [ ]:
import re

print("=== SICHERHEITSPRÜFUNG TAG-09 ===")

muster = {
    "E-Mail-Adressen": r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}",
    "GitHub-Tokens": r"gh[pousr]_[A-Za-z0-9_]{20,}",
    "Private Schlüssel": r"-----BEGIN (?:RSA |EC |OPENSSH )?PRIVATE KEY-----",
    "AWS-Zugangsschlüssel": r"AKIA[0-9A-Z]{16}"
}

treffer_gesamt = 0

for datei in sorted(abgabe.iterdir()):
    if not datei.is_file():
        continue

    inhalt = datei.read_text(encoding="utf-8")
    print(f"\nDatei: {datei.name}")

    for bezeichnung, muster_text in muster.items():
        anzahl = len(re.findall(muster_text, inhalt))
        treffer_gesamt += anzahl
        print(f"  {bezeichnung}: {anzahl} Treffer")

assert treffer_gesamt == 0, "Mögliche sensible Daten gefunden!"

print("\n✅ Keine Treffer bei den geprüften Mustern!")

In [ ]:
import ast

# Erwartete Abgabedateien
erwartet = {
    "README.md",
    "clean_bestellungen.py",
    "requirements.txt"
}

vorhanden = {datei.name for datei in abgabe.iterdir()}

# Dateiliste prüfen
assert vorhanden == erwartet, (
    f"Unerwartete Dateien: {vorhanden.symmetric_difference(erwartet)}"
)

# Python-Syntax prüfen, ohne das Skript auszuführen
skript_text = (abgabe / "clean_bestellungen.py").read_text(
    encoding="utf-8"
)

ast.parse(skript_text)

print("✅ Genau drei Abgabedateien vorhanden")
print("✅ Python-Syntax gültig")
print("✅ Keine CSV-Dateien im Abgabeordner")
print("✅ Abgabe technisch geprüft")

In [ ]:
import tempfile
import shutil
import subprocess
import hashlib
import sys
from pathlib import Path

with tempfile.TemporaryDirectory() as temp:
    testordner = Path(temp)

    # Frische Projektstruktur erstellen
    (testordner / "raw").mkdir()
    (testordner / "clean").mkdir()

    # Benötigte Dateien kopieren
    shutil.copy2(projekt / "clean_bestellungen.py", testordner)
    shutil.copy2(projekt / "requirements.txt", testordner)
    shutil.copy2(
        projekt / "raw" / "bestellungen_april_roh.csv",
        testordner / "raw"
    )

    hashes = []

    for durchlauf in range(1, 3):
        subprocess.run(
            [sys.executable, "clean_bestellungen.py"],
            cwd=testordner,
            check=True,
            capture_output=True,
            text=True
        )

        ausgabe = testordner / "clean" / "bestellungen_april_clean.csv"

        hashwert = hashlib.sha256(ausgabe.read_bytes()).hexdigest()
        hashes.append(hashwert)

        print(f"Durchlauf {durchlauf}: {hashwert}")

    assert hashes[0] == hashes[1], "Hashes unterscheiden sich!"

    print("\n✅ Beide Durchläufe erfolgreich")
    print("✅ SHA-256-Hashes identisch")
    print("✅ Test im frischen Verzeichnis bestanden")

In [ ]:
# Reproduzierbarkeitstest dokumentieren
zusatz = """
## Test im frischen Verzeichnis

Die Pipeline wurde in einem neu erstellten temporären
Projektverzeichnis zweimal ausgeführt.

Beide Durchläufe waren erfolgreich.

SHA-256 Durchlauf 1:
44b8afcc667c74a6da5b765b7732ae34d1c518982b5bd8c0a11f5b1d705862f9

SHA-256 Durchlauf 2:
44b8afcc667c74a6da5b765b7732ae34d1c518982b5bd8c0a11f5b1d705862f9

Ergebnis: Beide Ausgabedateien sind bytegenau identisch.

Hinweis: Der Test verwendete die bereits installierte
Python-Umgebung.
"""

readme = projekt / "README.md"

inhalt = readme.read_text(encoding="utf-8")

if "## Test im frischen Verzeichnis" not in inhalt:
    readme.write_text(inhalt + "\n" + zusatz, encoding="utf-8")

# Aktualisierte README in den Abgabeordner kopieren
shutil.copy2(readme, abgabe / "README.md")

print("✅ Reproduzierbarkeitstest dokumentiert")
print("✅ README im Abgabeordner aktualisiert")

In [ ]:
import json

# Original-Notebook einlesen
notebook_pfad = projekt.parent / "Woche2_Tag4.ipynb"

with open(notebook_pfad, "r", encoding="utf-8") as f:
    notebook = json.load(f)

# Gespeicherte Ausgaben aus der Kopie entfernen
for zelle in notebook["cells"]:
    if zelle["cell_type"] == "code":
        zelle["outputs"] = []
        zelle["execution_count"] = None

# Kopie im Abgabeordner speichern
ziel = abgabe / "Woche2_Tag4.ipynb"

with open(ziel, "w", encoding="utf-8") as f:
    json.dump(notebook, f, ensure_ascii=False, indent=1)

print("✅ Notebook-Kopie erstellt:", ziel.name)
print("✅ Zellenausgaben aus der Kopie entfernt")
print("✅ Original-Notebook unverändert")